<a href="https://colab.research.google.com/github/deekaykay07-hub/aether/blob/main/workingbackendforhermes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import time

print("Stay-alive loop started. The tunnel and proxy will remain active as long as this cell runs.")
try:
    while True:
        time.sleep(60)
        print(f"Heartbeat: {time.strftime('%H:%M:%S')} - Services are active.")
except KeyboardInterrupt:
    print("Stop signal received. Keep-alive terminated.")

Stay-alive loop started. The tunnel and proxy will remain active as long as this cell runs.
Stop signal received. Keep-alive terminated.


In [2]:
import time

print("Stay-alive loop started. The tunnel and proxy will remain active as long as this cell runs.")
try:
    while True:
        time.sleep(60)
        print(f"Heartbeat: {time.strftime('%H:%M:%S')} - Services are active.")
except KeyboardInterrupt:
    print("Stop signal received. Keep-alive terminated.")

Stay-alive loop started. The tunnel and proxy will remain active as long as this cell runs.
Stop signal received. Keep-alive terminated.


## 🚀 Hermes-Ollama Master Controller
Run this cell to start the entire stack. It will:
1. Verify GPU availability.
2. Start the Ollama daemon.
3. Launch a protocol-compliant Flask proxy.
4. Create a public ngrok tunnel.

In [ ]:
import subprocess, time, threading, requests, os, json, sys
!pip install pyngrok
from pyngrok import ngrok, conf
from flask import Flask, request, Response, jsonify, stream_with_context

# --- CONFIGURATION ---
MODEL_NAME = "dolphin-llama3:8b"
PROXY_PORT = 5000
OLLAMA_PORT = 11434

# --- 1. ENVIRONMENT & GPU CHECK ---
print("-- Stage 1: Environment Setup ---")
!apt-get install -y -qq pciutils zstd

# Install Ollama
print("Installing Ollama...")
!curl -fsSL https://ollama.com/install.sh | sh

gpu_check = subprocess.run(["lspci"], capture_output=True, text=True)
if "nvidia" in gpu_check.stdout.lower():
    print("✅ NVIDIA GPU Detected.")
else:
    print("⚠️ No NVIDIA GPU found.")

# --- 2. CLEANUP & DAEMON START ---
print("\n--- Stage 2: Starting Ollama ---")
!pkill -9 ollama
!fuser -k {PROXY_PORT}/tcp > /dev/null 2>&1
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels: ngrok.disconnect(t.public_url)
except: pass

env = {**os.environ, "OLLAMA_HOST": f"127.0.0.1:{OLLAMA_PORT}"}
with open('ollama.log', 'w') as log_file:
    subprocess.Popen(["ollama", "serve"], env=env, stdout=log_file, stderr=log_file, preexec_fn=os.setpgrp)

# Improved Wait for API readiness
print("Waiting for Ollama API to initialize...")
model_ready = False
for i in range(60):
    try:
        # Check if daemon is up AND model is pulled/available
        r = requests.get(f"http://127.0.0.1:{OLLAMA_PORT}/api/tags")
        if r.status_code == 200:
            print("✅ Ollama Daemon is responding.")
            # Attempt to pre-load model
            requests.post(f"http://127.0.0.1:{OLLAMA_PORT}/api/show", json={"name": MODEL_NAME})
            model_ready = True
            break
    except:
        pass
    time.sleep(2)

if not model_ready:
    print("❌ Ollama failed to start within 2 minutes. Check ollama.log.")

# --- 3. PROTOCOL PROXY ---
app = Flask(__name__)

@app.route('/v1/models', methods=['GET'])
@app.route('/api/v1/models', methods=['GET'])
@app.route('/api/tags', methods=['GET'])
def list_models():
    return jsonify({"object": "list", "data": [{"id": MODEL_NAME, "object": "model", "created": int(time.time()), "owned_by": "ollama"}]})

@app.route('/v1/chat/completions', methods=['POST'])
def chat_proxy():
    data = request.get_json(force=True) or {}
    def generate():
        try:
            payload = {"model": MODEL_NAME, "messages": data.get("messages", []), "stream": True}
            with requests.post(f"http://127.0.0.1:{OLLAMA_PORT}/api/chat", json=payload, stream=True, timeout=120) as resp:
                for line in resp.iter_lines():
                    if not line: continue
                    chunk = json.loads(line)
                    content = chunk.get("message", {}).get("content", "")
                    yield f"data: {json.dumps({'id':'1','object':'chat.completion.chunk','choices':[{'index':0,'delta':{'content':content},'finish_reason':None}]})}\n\n".encode('utf-8')
                    if chunk.get("done", False):
                        yield f"data: {json.dumps({'choices':[{'index':0,'delta':{},'finish_reason':'stop'}]})}\n\n".encode('utf-8')
            yield "data: [DONE]\n\n".encode('utf-8')
        except Exception as e:
            yield f"data: {json.dumps({'error': f'Ollama Connection Error: {str(e)}'})}\n\n".encode('utf-8')

    if data.get("stream", False):
        return Response(stream_with_context(generate()), mimetype='text/event-stream')

    try:
        r = requests.post(f"http://127.0.0.1:{OLLAMA_PORT}/api/chat", json={**data, "stream": False}, timeout=120)
        o = r.json()
        return jsonify({"choices":[{"message":{"role":"assistant","content":o['message']['content']},"finish_reason":"stop"}]})
    except Exception as e:
        return jsonify({"error": f"Ollama connection failed: {str(e)}"}), 503

print("--- Stage 3: Launching Resilient Proxy ---")
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=PROXY_PORT, threaded=True), daemon=True).start()
time.sleep(2)

# --- 4. TUNNELING ---
print("--- Stage 4: Establishing Tunnel ---")
try:
    pub_url = ngrok.connect(PROXY_PORT).public_url
    print(f"\n🚀 MASTER STACK RECOVERED")
    print(f"🔗 New Public Endpoint: {pub_url}")
except Exception as e:
    print(f"❌ Tunnel Error: {e}")

-- Stage 1: Environment Setup ---
Installing Ollama...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
⚠️ No NVIDIA GPU found.

--- Stage 2: Starting Ollama ---


In [ ]:
print('--- Ollama Daemon Logs (Last 50 Lines) ---')
!tail -n 50 ollama.log

In [ ]:
print("Checking Ollama API health via curl...")
!curl -I http://127.0.0.1:11434/api/tags

### ⏳ Session Keep-Alive
Run this cell to prevent Colab from idling. Keep this browser tab open.

In [ ]:
import time

print("Stay-alive loop started. The tunnel and proxy will remain active as long as this cell runs.")
try:
    while True:
        time.sleep(60)
        print(f"Heartbeat: {time.strftime('%H:%M:%S')} - Services are active.")
except KeyboardInterrupt:
    print("Stop signal received. Keep-alive terminated.")

### 🔍 System Diagnostics
Run this to check if all services are still running.

In [3]:
import requests
import os
import subprocess

def check_status():
    print("--- Diagnostic Report ---")

    # 1. Check Ollama
    try:
        r = requests.get('http://127.0.0.1:11434/api/tags', timeout=2)
        print(f"✅ Ollama Daemon: Running (Status {r.status_code})")
    except:
        print("❌ Ollama Daemon: NOT RESPONDING")

    # 2. Check Proxy
    try:
        r = requests.get('http://127.0.0.1:5000/v1/models', timeout=2)
        print(f"✅ Flask Proxy: Running (Status {r.status_code})")
    except:
        print("❌ Flask Proxy: NOT RESPONDING")

    # 3. Check ngrok
    try:
        tunnels = ngrok.get_tunnels()
        if tunnels:
            print(f"✅ ngrok Tunnel: Active ({tunnels[0].public_url})")
        else:
            print("❌ ngrok Tunnel: NO ACTIVE TUNNELS")
    except Exception as e:
        print(f"❌ ngrok Check Failed: {e}")

check_status()

--- Diagnostic Report ---
❌ Ollama Daemon: NOT RESPONDING
❌ Flask Proxy: NOT RESPONDING


ERROR:pyngrok.process.ngrok:t=2026-07-29T21:02:36+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-07-29T21:02:36+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
CRITICAL:pyngrok.process.ngrok:t=2026-07-29T21:02:36+0000 lvl=crit msg="command failed" err="authentication failed: This ngrok session is not authenticated. ngrok requi

❌ ngrok Check Failed: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.


In [2]:
from pyngrok import ngrok

try:
    # Disconnect existing to be safe
    tunnels = ngrok.get_tunnels()
    for t in tunnels: ngrok.disconnect(t.public_url)

    # Start new tunnel
    pub_url = ngrok.connect(5000).public_url
    print(f"\n🚀 TUNNEL RE-ESTABLISHED")
    print(f"🔗 New Public Endpoint: {pub_url}")
    print(f"\nRun this on your local machine:")
    print(f"hermes config set openai_url {pub_url}/v1")
except Exception as e:
    print(f"❌ Tunnel Error: {e}")

ERROR:pyngrok.process.ngrok:t=2026-07-29T21:02:29+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-07-29T21:02:29+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"


❌ Tunnel Error: The ngrok process errored on start: authentication failed: This ngrok session is not authenticated. ngrok requires an account and a valid credential to start a session.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nGet your credential: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.


In [9]:
print("--- Testing Proxy via cURL ---")
!curl -X POST http://127.0.0.1:5000/v1/chat/completions \
     -H "Content-Type: application/json" \
     -d '{"model": "dolphin-llama3:8b", "messages": [{"role": "user", "content": "Say hello!"}], "stream": false}'

INFO:werkzeug:127.0.0.1 - - [29/Jul/2026 21:01:20] "POST /v1/chat/completions HTTP/1.1" 503 -


--- Testing Proxy via cURL ---
{"error":"Ollama connection failed: 'message'"}


In [10]:
print("--- Testing Patched Streaming Proxy via cURL ---")
# Testing with the new direct_passthrough settings
!curl -N -X POST http://127.0.0.1:5000/v1/chat/completions \
     -H "Content-Type: application/json" \
     -d '{"model": "dolphin-llama3:8b", "messages": [{"role": "user", "content": "Verify stream stability with a short sentence."}], "stream": true}'

INFO:werkzeug:127.0.0.1 - - [29/Jul/2026 21:01:24] "POST /v1/chat/completions HTTP/1.1" 200 -


--- Testing Patched Streaming Proxy via cURL ---
data: {"id": "1", "object": "chat.completion.chunk", "choices": [{"index": 0, "delta": {"content": ""}, "finish_reason": null}]}

data: [DONE]



In [ ]:
print("--- Final Verification: Streaming Test ---")
import requests
import json

url = "http://127.0.0.1:5000/v1/chat/completions"
payload = {
    "model": "dolphin-llama3:8b",
    "messages": [{"role": "user", "content": "Respond with 'Streaming is working!'"}],
    "stream": True
}

try:
    response = requests.post(url, json=payload, stream=True, timeout=10)
    print(f"Response Status: {response.status_code}")
    for line in response.iter_lines():
        if line:
            print(line.decode('utf-8'))
except Exception as e:
    print(f"❌ Verification Failed: {e}")